In [3]:
import torch
import os
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
import psycopg2
from psycopg2.extras import execute_batch

# 1. Kiểm tra kết nối GPU đời 50
try:
    print(f"Bản Torch: {torch.__version__}")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Thiết bị đang sử dụng: {device}")
    if torch.cuda.is_available():
        print(f"Đang dùng quái vật: {torch.cuda.get_device_name(0)}")
except Exception as e:
    print(f"Lỗi khởi động Torch: {e}")

# 2. Khởi tạo Model (Tối ưu cho VRAM 16GB)
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

# 3. Kết nối Database
conn = psycopg2.connect(dbname="KanvaPro", user="postgres", password="Superhero3@", host="localhost")
cur = conn.cursor()

# Lấy các sticker cần dán nhãn
cur.execute("SELECT id, url FROM assets WHERE type = 'sticker' AND (name ILIKE '%premium%' OR name = 'Sticker')")
rows = cur.fetchall()

storage_base_path = "E:/DoAn2026/KanvaPro/backend/sticker_upload"
BATCH_SIZE = 64 # Card 5060 Ti 16GB dư sức gánh 64 ảnh một lúc

print(f"Bắt đầu xử lý {len(rows)} ảnh...")

for i in range(0, len(rows), BATCH_SIZE):
    batch = rows[i:i + BATCH_SIZE]
    images = []
    valid_ids = []

    for asset_id, url in batch:
        img_path = os.path.join(storage_base_path, url.lstrip('/'))
        if os.path.exists(img_path):
            try:
                images.append(Image.open(img_path).convert('RGB'))
                valid_ids.append(asset_id)
            except: continue

    if not images: continue

    # Chạy AI hàng loạt
    inputs = processor(images=images, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=30)
    
    captions = processor.batch_decode(outputs, skip_special_tokens=True)

    # Cập nhật DB theo lô
    update_data = [(caption.capitalize(), aid) for caption, aid in zip(captions, valid_ids)]
    execute_batch(cur, "UPDATE assets SET name = %s WHERE id = %s", update_data)
    conn.commit()
    
    print(f"Đã xử lý xong: {i + len(images)}/{len(rows)}")

cur.close()
conn.close()
print("Dán nhãn hoàn tất bằng RTX 5060 Ti!")

Bản Torch: 2.10.0+cpu
Thiết bị đang sử dụng: cpu


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

Bắt đầu xử lý 5732 ảnh...
Đã xử lý xong: 64/5732
Đã xử lý xong: 128/5732
Đã xử lý xong: 192/5732
Đã xử lý xong: 256/5732
Đã xử lý xong: 320/5732
Đã xử lý xong: 384/5732
Đã xử lý xong: 448/5732
Đã xử lý xong: 512/5732
Đã xử lý xong: 576/5732
Đã xử lý xong: 640/5732
Đã xử lý xong: 704/5732
Đã xử lý xong: 768/5732
Đã xử lý xong: 832/5732
Đã xử lý xong: 896/5732
Đã xử lý xong: 960/5732
Đã xử lý xong: 1024/5732
Đã xử lý xong: 1088/5732
Đã xử lý xong: 1152/5732
Đã xử lý xong: 1216/5732
Đã xử lý xong: 1280/5732
Đã xử lý xong: 1344/5732
Đã xử lý xong: 1408/5732
Đã xử lý xong: 1472/5732
Đã xử lý xong: 1536/5732
Đã xử lý xong: 1600/5732
Đã xử lý xong: 1664/5732
Đã xử lý xong: 1728/5732
Đã xử lý xong: 1792/5732
Đã xử lý xong: 1856/5732
Đã xử lý xong: 1920/5732
Đã xử lý xong: 1984/5732
Đã xử lý xong: 2048/5732
Đã xử lý xong: 2112/5732
Đã xử lý xong: 2176/5732
Đã xử lý xong: 2240/5732
Đã xử lý xong: 2304/5732
Đã xử lý xong: 2368/5732
Đã xử lý xong: 2432/5732
Đã xử lý xong: 2496/5732
Đã xử lý xong: 

In [2]:
%pip install torch transformers psycopg2-binary Pillow

  Using cached torch-2.10.0-cp313-cp313-win_amd64.whl.metadata (31 kB)
  Using cached psycopg2_binary-2.9.11-cp313-cp313-win_amd64.whl.metadata (5.1 kB)
Using cached torch-2.10.0-cp313-cp313-win_amd64.whl (113.8 MB)
Using cached psycopg2_binary-2.9.11-cp313-cp313-win_amd64.whl (2.7 MB)
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
